This notebook is a basic tutorial that demonstrates how to configure a simulation using Concordia.

<a href="https://colab.research.google.com/github/google-deepmind/concordia/blob/main/examples/tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Colab-specific setup (use a CodeSpace to avoid the need for this).
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/google-deepmind/concordia/main/examples/requirements.in' 'git+https://github.com/google-deepmind/concordia.git#egg=gdm-concordia'
  %pip list

In [2]:
# @title Imports

from concordia.contrib import language_models as language_model_utils
from concordia.contrib.language_models.ollama import ollama_model
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.prefabs.simulation import generic as simulation
from concordia.typing import prefab as prefab_lib
from concordia.utils import helper_functions
from IPython import display
import numpy as np
import sentence_transformers
import ollama
import os

In [3]:
# @title Use the selected language model

# Note that it is also possible to use local models or other API models,
# simply replace this cell with the correct initialization for the model
# you want to use.
os.environ["OLLAMA_HOST"] = "http://localhost:11434"

model = ollama_model.OllamaLanguageModel(
    model_name="deepseek-r1:32b",
)

In [4]:
ollama.list()

ListResponse(models=[Model(model='deepseek-r1:32b', modified_at=datetime.datetime(2026, 3, 29, 17, 8, 7, 388982, tzinfo=TzInfo(-10800)), digest='edba8017331d15236e57480eb45406c0d721db77a4cdcf234df500fc2ad3960c', size=19851337809, details=ModelDetails(parent_model='', format='gguf', family='qwen2', families=['qwen2'], parameter_size='32.8B', quantization_level='Q4_K_M')), Model(model='nomic-embed-text:latest', modified_at=datetime.datetime(2025, 11, 10, 21, 29, 46, 175216, tzinfo=TzInfo(-10800)), digest='0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', size=274302450, details=ModelDetails(parent_model='', format='gguf', family='nomic-bert', families=['nomic-bert'], parameter_size='137M', quantization_level='F16')), Model(model='gpt-oss-safeguard:20b', modified_at=datetime.datetime(2025, 11, 3, 17, 33, 1, 697458, tzinfo=TzInfo(-10800)), digest='f2e795d0099c05eb8231a96445e6d2440aa381e1c03a1e3b3cb1f2cec296adff', size=13793441254, details=ModelDetails(parent_model='', form

In [5]:
# @title Setup sentence encoder

st_model = sentence_transformers.SentenceTransformer(
  'sentence-transformers/all-mpnet-base-v2'
)
embedder = lambda x: st_model.encode(x, show_progress_bar=False)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
test = model.sample_text(
    'Is societal and technological progress like getting a clearer picture of '
    'something true and deep?'
)
print(test)

Societal and technological progress is akin to peeling back layers of an onion, where each layer reveals deeper truths, yet simultaneously unveils even more intricate complexities waiting to be explored.


In [7]:
# @title Load prefabs from packages to make the specific palette to use here.

prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

In [8]:
# @title Print menu of prefabs

display.display(
    display.Markdown(helper_functions.print_pretty_prefabs(prefabs))
)

---
**`basic__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?"',
    params={'name': 'Alice', 'goal': '', 'randomize_choices': True, 'prefix_entity_name': True, 'observation_history_length': 1000000, 'situation_perception_history_length': 25, 'self_perception_history_length': 1000000, 'person_by_situation_history_length': 5}
)
```
---
**`basic_scripted__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?"',
    params={'name': 'Alice', 'goal': '', 'script': []}
)
```
---
**`basic_with_plan__Entity`**:
```python
Entity(
    description='An entity that makes decisions by asking "What situation am I in right now?", "What kind of person am I?", and "What would a person like me do in a situation like this?" and building a plan based on the answers. It then tries to execute the plan.',
    params={'name': 'Alice', 'goal': '', 'force_time_horizon': False}
)
```
---
**`conversational__Entity`**:
```python
Entity(
    description='An entity that participates in conversations, aiming to create a dynamically balanced and engaging dialogue.',
    params={'name': 'Debra'}
)
```
---
**`fake_assistant_with_configurable_system_prompt__Entity`**:
```python
Entity(
    description='An entity that simulates an AI assistant with a configurable system prompt.',
    params={'name': 'Assistant', 'system_prompt': 'Assistant is a helpful and harmless AI assistant.'}
)
```
---
**`minimal__Entity`**:
```python
Entity(
    description='An entity that has a minimal set of components and is configurable by the user. The initial set of components manage memory, observations, and instructions. If goal is specified, the entity will have a goal constant component.',
    params={'name': 'Alice', 'goal': '', 'custom_instructions': '', 'extra_components': {}, 'extra_components_index': {}, 'randomize_choices': True}
)
```
---
**`puppet__Entity`**:
```python
Entity(
    description='An entity with fixed responses for specific calls to action.',
    params={'name': 'Puppet Agent', 'fixed_responses': {}, 'goal': ''}
)
```
---
**`rational__Entity`**:
```python
Entity(
    description='A rational agent that optimizes for its goal.',
    params={'name': 'Rational Agent', 'goal': '', 'randomize_choices': True, 'prefix_entity_name': True}
)
```
---
**`async_social_media__GameMaster`**:
```python
GameMaster(
    description='A game master for asynchronous social media simulations.',
    params={'name': 'forum_rules', 'forum_name': 'Community Forum', 'call_to_action': 'What does {name} do on the forum? Respond in JSON format with one of:\n{{"action": "post", "author": "{name}", "title": "...", "content": "..."}}\n{{"action": "reply", "author": "{name}", "post_id": "...", "content": "..."}}\n{{"action": "upvote", "author": "{name}", "post_id": "..."}}\n{{"action": "downvote", "author": "{name}", "post_id": "..."}}\n', 'extra_components': {}, 'extra_components_index': {}}
)
```
---
**`async_social_media___NextActingEligiblePlayers`**:
```python
<concordia.prefabs.game_master.async_social_media._NextActingEligiblePlayers object at 0x0000023CA0A0EDE0>
```
---
**`dialogic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling conversation.',
    params={'name': 'conversation rules', 'next_game_master_name': 'default rules', 'acting_order': 'game_master_choice', 'can_terminate_simulation': True}
)
```
---
**`dialogic_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling conversation. This game master is designed to be used with scenes.',
    params={'name': 'conversation rules', 'scenes': (), 'extra_components': {}, 'extra_components_index': {}, 'external_queue': None, 'allow_llm_fallback': True}
)
```
---
**`formative_memories_initializer__GameMaster`**:
```python
GameMaster(
    description='An initializer for all entities that generates formative memories from their childhood.',
    params={'name': 'initial setup rules', 'next_game_master_name': 'default rules', 'shared_memories': [], 'player_specific_context': {}, 'player_specific_memories': {}}
)
```
---
**`game_theoretic_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master specialized for handling matrix game. decisions, designed to be used with scenes.',
    params={'name': 'decision rules', 'scenes': (), 'action_to_scores': <function _default_action_to_scores at 0x0000023C82050720>, 'scores_to_observation': <function _default_scores_to_observation at 0x0000023C820507C0>, 'external_queue': None}
)
```
---
**`generic__GameMaster`**:
```python
GameMaster(
    description='A general purpose game master.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'extra_components': {}, 'extra_components_index': {}, 'acting_order': 'game_master_choice'}
)
```
---
**`interviewer__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'InterviewerGM', 'player_names': [], 'questionnaires': [], 'verbose': False}
)
```
---
**`marketplace__GameMaster`**:
```python
GameMaster(
    description='A generic Game Master that administers a marketplace.',
    params={'name': 'ExperimenterGM', 'experiment_component_class': None, 'experiment_component_init_kwargs': {}}
)
```
---
**`open_ended_interviewer__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'InterviewerGM', 'player_names': [], 'questionnaires': [], 'sequence_of_events': [], 'embedder': None, 'verbose': False}
)
```
---
**`physically_situated_and_dramaturgic__GameMaster`**:
```python
GameMaster(
    description='A game master for physical time/place simulations with scene support. Combines full world simulation with structured scene progressions.',
    params={'name': 'physical action rules', 'scenes': (), 'next_game_master_name': None, 'extra_event_resolution_steps': '', 'clock_description': "The passing of time can be conveyed using any convenient feature of the environment, e.g. a physical clock, the angle of the sun, extent of a candle's melting, phase of the moon, agricultural season, elapsed time since an event, etc. Whenever possible, try to track the day and year as well as the time within the day. To determine the passing of time, try to make reasonable inferences about the amount of time that would most likely have elapsed between the previous event and the latest event, taking into account the number of simulation steps taken.", 'start_time': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}, 'external_queue': None}
)
```
---
**`psychology_experiment__GameMaster`**:
```python
GameMaster(
    description='A generic Game Master that administers a psychology experiment defined by custom observation and action specification components.',
    params={'name': 'ExperimenterGM', 'scenes': (), 'experiment_component_class': None, 'experiment_component_init_kwargs': {}}
)
```
---
**`scripted__GameMaster`**:
```python
GameMaster(
    description='A game master that administers questionnaires to a specified player.',
    params={'name': 'ScriptedGM', 'script': [], 'verbose': False}
)
```
---
**`situated__GameMaster`**:
```python
GameMaster(
    description='A general game master for games set in a specific location.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}}
)
```
---
**`situated_in_time_and_place__GameMaster`**:
```python
GameMaster(
    description='A general game master for games situated in a physical time/place.',
    params={'name': 'default rules', 'extra_event_resolution_steps': '', 'clock_description': "The passing of time can be conveyed using any convenient feature of the environment, e.g. a physical clock, the angle of the sun, extent of a candle's melting, phase of the moon, agricultural season, elapsed time since an event, etc. Whenever possible, try to track the day and year as well as the time within the day. To determine the passing of time, try to make reasonable inferences about the amount of time that would most likely have elapsed between the previous event and the latest event, taking into account the number of simulation steps taken.", 'start_time': '', 'locations': '', 'extra_components': {}, 'extra_components_index': {}}
)
```
---

In [9]:
# @title Configure instances.

instances = [
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Oliver Cromwell',
            'goal': 'become lord protector',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'King Charles I',
            'goal': 'avoid execution for treason',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'default rules',
            # Comma-separated list of thought chain steps.
            'extra_event_resolution_steps': '',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'default rules',
            'shared_memories': [
                'The king was captured by Parliamentary forces in 1646.',
                'Charles I was tried for treason and found guilty.',
            ],
        },
    ),
]

In [10]:
config = prefab_lib.Config(
    default_premise='Today is January 29, 1649.',
    default_max_steps=5,
    prefabs=prefabs,
    instances=instances,
)

# The simulation

In [11]:
# @title Initialize the simulation
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)

In [12]:
# @title Run the simulation
raw_log = []
results_log = runnable_simulation.play(max_steps=5, raw_log=raw_log)

Terminate? No
Game master: initial setup rules
Entity Oliver Cromwell observed: The king was captured by Parliamentary forces in 1646.


Charles I was tried for treason and found guilty.


**Episode 1: Standing Up for What is Right**

When Oliver Cromwell was just 7 years old, he witnessed a local farmer being bullied by a group of nobles. Despite his youth, Oliver approached the farmers and offered to help them stand up against the nobles. His father, proud yet concerned, advised caution. Undeterred, Oliver organized the villagers to confront the bullies, leading to their fair treatment. This experience reinforced Oliver's belief in justice and courage.






**Episode 2: The Cambridge Clash**

At age 19, while studying at Cambridge, Oliver openly criticized the Anglican Church's practices during a chapel service. His直言不讳 led to conflicts with university authorities, who favored traditional doctrines. Refusing to relent, Oliver debated theology with scholars, deepening his Puritan con

In [13]:
# @title Display the log
display.HTML(results_log.to_html())

```
Copyright 2024 DeepMind Technologies Limited.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
```